# 第04回 機械学習2：値とコードを一段ずつ読む

距離と投票、分岐と混ざり具合、モデル間の多数決を同じ固定事例で追います。SVMはスライド・操作教材で補います。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** 問い合わせ(2,3)ではk1がB・k3がA。分岐候補の加重Giniを比べ、3モデルの票を数える。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
from collections import Counter


In [ ]:
from fractions import Fraction


In [ ]:
import json


In [ ]:
TRAIN = (("T1", 0, 0, "A"), ("T2", 0, 2, "A"),
         ("T3", 2, 0, "A"), ("T4", 2, 2, "B"),
         ("T5", 6, 6, "B"), ("T6", 6, 8, "B"), ("T7", 8, 6, "B"))


In [ ]:
VALIDATION = (("V1", 2, 3, "A"), ("V2", 3, 2, "A"),
              ("V3", 7, 7, "B"), ("V4", 8, 8, "B"))


In [ ]:
TEST = (("E1", 1, 1, "A"), ("E2", 3, 3, "A"),
        ("E3", 7, 6, "B"), ("E4", 6, 7, "B"))


In [ ]:
QUERY = (2, 3)


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 近傍の選択

**今いる段階：** 差→距離→順位→投票

**予想：** どの点が投票するか

**操作：** example_kを3から1へ変更し、同じセルを実行します。

**結果の目印：** 距離順と採用k件が対応する

**戻す：** example_k=3に戻して再実行します。

**説明する：** 距離とラベルの役割


### 計算の定義：`validate_rows`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def validate_rows(rows):
    if not rows:
        raise ValueError("データは1件以上必要です")
    ids = []
    for row in rows:
        if len(row) != 4:
            raise ValueError("行はID・x・y・ラベルの4項目です")
        identity, x, y, label = row
        if not isinstance(identity, str) or not identity:
            raise ValueError("IDは空でない文字列です")
        if type(x) is not int or type(y) is not int or label not in ("A", "B"):
            raise ValueError("特徴は整数、ラベルはAまたはBです")
        ids.append(identity)
    if len(ids) != len(set(ids)):
        raise ValueError("IDが重複しています")


### 計算の定義：`validate_point`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def validate_point(point):
    if len(point) != 2 or any(type(value) is not int for value in point):
        raise ValueError("入力は整数の特徴2個です")


### 計算の定義：`nearest`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def nearest(train, point, k, scales=(1, 1)):
    validate_rows(train)
    validate_point(point)
    if type(k) is not int or not 1 <= k <= len(train):
        raise ValueError("kは1以上、学習件数以下の整数です")
    if len(scales) != 2 or any(type(s) is not int or s <= 0 for s in scales):
        raise ValueError("尺度は正の整数2個です")
    ranked = []
    for identity, x, y, label in train:
        d2 = Fraction((x - point[0]) ** 2, scales[0] ** 2)
        d2 += Fraction((y - point[1]) ** 2, scales[1] ** 2)
        ranked.append((d2, identity, label))
    # 距離が等しい場合はID順。元のTRAINを書き換えない。
    return sorted(ranked)[:k]


### 計算の定義：`majority`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def majority(labels):
    if not labels or any(label not in ("A", "B") for label in labels):
        raise ValueError("投票は1票以上、AまたはBです")
    counts = Counter(labels)
    # 同票ならA。この教材の規約であり、普遍的な規則ではない。
    return min(counts, key=lambda label: (-counts[label], label))


### 計算の定義：`predict_knn`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def predict_knn(train, point, k, scales=(1, 1)):
    return majority([label for _, _, label in nearest(train, point, k, scales)])


### 計算の定義：`lesson_neighbors`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_neighbors(k=3):
    all_neighbors = nearest(TRAIN,QUERY,len(TRAIN))
    chosen = nearest(TRAIN,QUERY,k)
    coordinates = {r[0]:r[1:3] for r in TRAIN}
    rows = []
    for rank,(distance,identity,label) in enumerate(all_neighbors,1):
        x,y=coordinates[identity]
        rows.append((rank,identity,x-QUERY[0],y-QUERY[1],str(distance),label,rank<=k))
    return {'title':'距離→近いk件→投票','question':'最も近い1件と近い3件で同じ予測か',
            'input':f'問い合わせ点{QUERY}、k={k}',
            'columns':['順位','ID','xの差','yの差','距離の二乗','ラベル','投票に使う'],'rows':rows,
            'result':{'投票':[r[2] for r in chosen],'予測':predict_knn(TRAIN,QUERY,k)},
            'comparison':'k1はB、k3はA。距離同点ならID順、投票同数ならA。',
            'explanation':'距離の二乗=(x差)^2+(y差)^2。平方根を取らなくても順位は同じ。正解ラベルは距離計算に使わない。',
            'reset':'k=3で再実行。'}


In [ ]:
example_k = 3
program_observation_neighbors = lesson_neighbors(example_k)
learning_show(program_observation_neighbors, reset='example_k=3に戻して再実行します。')


## 2. 尺度と距離

**今いる段階：** 座標差→尺度→距離

**予想：** 単位を変えると近さはどうなるか

**操作：** example_y_scaleを1から3へ変更し、同じセルを実行します。

**結果の目印：** S1の距離二乗9→1

**戻す：** example_y_scale=1に戻して再実行します。

**説明する：** 距離の各項への寄与


### 計算の定義：`lesson_scale`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_scale(y_scale=1):
    rows_data = (('S1',0,3,'A'),('S2',2,0,'B'))
    ranked = nearest(rows_data,(0,0),2,(1,y_scale))
    return {'title':'尺度をそろえると近さが変わる','question':'同じ座標なら近い点は変わらないか',
            'input':f'問い合わせ(0,0)、yを{y_scale}で割って距離を計算。',
            'columns':['順位','ID','距離の二乗','ラベル'],'rows':[(i+1,r[1],str(r[0]),r[2]) for i,r in enumerate(ranked)],
            'tables':[('差を尺度で割る',['ID','x差','y差','x差/1','y差/尺度'],[(r[0],r[1],r[2],r[1],str(Fraction(r[2],y_scale))) for r in rows_data])],
            'result':predict_knn(rows_data,(0,0),1,(1,y_scale)),
            'comparison':'y_scale1→3で、S1の距離二乗9→1。S2の4より近くなる。',
            'explanation':'特徴の単位が異なると距離への寄与が偏る。尺度の決め方は学習側で定め、テストの正解を見て調整しない。',
            'reset':'y_scale=1で再実行。'}


In [ ]:
example_y_scale = 1
program_observation_scale = lesson_scale(example_y_scale)
learning_show(program_observation_scale, reset='example_y_scale=1に戻して再実行します。')


## 3. 1段決定木

**今いる段階：** 候補→振り分け→不純度→選択

**予想：** どの分岐が混ざりを減らすか

**操作：** example_t4を'B'から'A'へ変更し、同じセルを実行します。

**結果の目印：** 候補ごとの左右の事例と加重Giniが表示される

**戻す：** example_t4='B'に戻して再実行します。

**説明する：** 件数で重み付けする理由


### 計算の定義：`gini`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def gini(rows):
    validate_rows(rows)
    counts = Counter(row[3] for row in rows)
    return 1 - sum(Fraction(n, len(rows)) ** 2 for n in counts.values())


### 計算の定義：`fit_stump`

候補の加重Giniが最小となる1回の分岐を学習側だけで選ぶ。


In [ ]:
def fit_stump(train, record=None):
    """候補の加重Giniが最小となる1回の分岐を学習側だけで選ぶ。"""
    validate_rows(train)
    base = majority([row[3] for row in train])
    candidates = []
    for axis in (0, 1):
        values = sorted({row[axis + 1] for row in train})
        for low, high in zip(values, values[1:]):
            threshold = Fraction(low + high, 2)
            left = [row for row in train if row[axis + 1] <= threshold]
            right = [row for row in train if row[axis + 1] > threshold]
            weighted = (len(left) * gini(left) + len(right) * gini(right)) / len(train)
            candidates.append((weighted, axis, threshold, left, right))
            if record is not None:
                record.append((axis, threshold, list(left), list(right), gini(left), gini(right), weighted))
    if not candidates:
        return {"axis": None, "threshold": None, "left": base, "right": base,
                "weighted_gini": gini(train)}
    # 同じ不純度ならx優先、次に小さいしきい値。改善しなければ分岐しない。
    weighted, axis, threshold, left, right = min(candidates, key=lambda c: c[:3])
    if weighted >= gini(train):
        return {"axis": None, "threshold": None, "left": base, "right": base,
                "weighted_gini": gini(train)}
    return {"axis": axis, "threshold": threshold,
            "left": majority([r[3] for r in left]),
            "right": majority([r[3] for r in right]), "weighted_gini": weighted}


### 計算の定義：`lesson_tree`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_tree(label_t4='B'):
    train = list(TRAIN)
    train[3] = (*train[3][:3],label_t4)
    trace = []
    model = fit_stump(train,record=trace)
    rows,tables = [],[]
    for axis,t,left,right,gl,gr,g in trace:
        rows.append(('xy'[axis],str(t),len(left),str(gl),len(right),str(gr),str(g)))
        tables.append((f'{"xy"[axis]}≤{t}の振り分け',['側','ID','ラベル'],[(side,r[0],r[3]) for side,data in [('左',left),('右',right)] for r in data]))
    return {'title':'分岐候補ごとの混ざり具合を比べる','question':'片方だけ純粋なら良い分岐か',
            'input':f'T4のラベル={label_t4}。学習7件をコピー。',
            'columns':['軸','しきい値','左件数','左Gini','右件数','右Gini','加重Gini'],'rows':rows,'tables':tables,
            'result':model,'comparison':'T4のラベルB→Aで同じ候補の混ざり具合と選択結果を比較する。',
            'explanation':'Gini=1−(A割合)^2−(B割合)^2。加重値=(左件数×左Gini＋右件数×右Gini)/全件数。同点ならx優先、次に小さいしきい値。1段の決定木である。',
            'reset':'label_t4="B"で再実行。TRAINは変更しない。'}


In [ ]:
example_t4 = 'B'
program_observation_tree = lesson_tree(example_t4)
learning_show(program_observation_tree, reset="example_t4='B'に戻して再実行します。")


## 4. モデルの多数決

**今いる段階：** 各予測→票数→多数決

**予想：** 誰が何票入れるか

**操作：** example_pointを(2, 3)から(7, 7)へ変更し、同じセルを実行します。

**結果の目印：** 3モデルの予測とA/B票数が対応する

**戻す：** example_point=(2, 3)に戻して再実行します。

**説明する：** 集約しても必ず正解とは限らない理由


### 計算の定義：`predict_stump`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def predict_stump(model, point):
    validate_point(point)
    if model["axis"] is None or point[model["axis"]] <= model["threshold"]:
        return model["left"]
    return model["right"]


### 計算の定義：`lesson_votes`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_votes(point=(2,3)):
    stump = fit_stump(TRAIN)
    votes = [predict_knn(TRAIN,point,1),predict_knn(TRAIN,point,3),predict_stump(stump,point)]
    return {'title':'三つの予測を一票ずつ集める','question':'三つの予測が異なると何を返すか',
            'input':list(point),'columns':['モデル','投票'],'rows':list(zip(['k=1','k=3','1段決定木'],votes)),
            'tables':[('票の集計',['ラベル','票数'],[(label,votes.count(label)) for label in ('A','B')])],
            'result':majority(votes),'comparison':'点(2,3)→(7,7)で各モデルの票を比較する。',
            'explanation':'この教材は異なる3モデルの多数決。ランダムフォレストやSVMの実装ではない。多数決でも各モデルが同じ誤りをすれば誤答する。',
            'reset':'point=(2,3)で再実行。'}


In [ ]:
example_point = (2, 3)
program_observation_votes = lesson_votes(example_point)
learning_show(program_observation_votes, reset='example_point=(2, 3)に戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`evaluate`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def evaluate(predictor, rows):
    validate_rows(rows)
    predictions = [predictor((r[1], r[2])) for r in rows]
    return {"predictions": predictions,
            "correct": sum(pred == row[3] for pred, row in zip(predictions, rows)),
            "total": len(rows)}


### 計算の定義：`choose_k`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def choose_k(validation_scores):
    if not validation_scores:
        raise ValueError("検証結果が必要です")
    # 比較する候補は同じ検証データを使う。同成績なら小さいk。
    return min(validation_scores, key=lambda k: (
        -Fraction(validation_scores[k]["correct"], validation_scores[k]["total"]), k))


### 計算の定義：`run_experiment`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def run_experiment():
    for rows in (TRAIN, VALIDATION, TEST):
        validate_rows(rows)
    ids = [row[0] for rows in (TRAIN, VALIDATION, TEST) for row in rows]
    if len(ids) != len(set(ids)):
        raise ValueError("分割間でIDが重複しています")
    train_scores = {k: evaluate(lambda point: predict_knn(TRAIN, point, k), TRAIN)
                    for k in (1, 3)}
    validation_scores = {k: evaluate(lambda point: predict_knn(TRAIN, point, k), VALIDATION)
                         for k in (1, 3)}
    selected = choose_k(validation_scores)
    # テストはkを確定した後、選んだモデルだけに使う。
    test_score = evaluate(lambda point: predict_knn(TRAIN, point, selected), TEST)
    stump = fit_stump(TRAIN)
    votes = [predict_knn(TRAIN, QUERY, 1), predict_knn(TRAIN, QUERY, 3),
             predict_stump(stump, QUERY)]
    scale_rows = (("S1", 0, 3, "A"), ("S2", 2, 0, "B"))
    return {
        "data": {"train": [list(r) for r in TRAIN],
                 "validation": [list(r) for r in VALIDATION], "test": [list(r) for r in TEST]},
        "query": list(QUERY),
        "nearest3": [{"id": i, "distance_squared": str(d), "label": label}
                     for d, i, label in nearest(TRAIN, QUERY, 3)],
        "train_scores": {str(k): score for k, score in train_scores.items()},
        "validation_scores": {str(k): score for k, score in validation_scores.items()},
        "selected_k": selected, "selected_test": test_score,
        "stump": {**stump, "threshold": str(stump["threshold"]),
                  "weighted_gini": str(stump["weighted_gini"]),
                  "query_prediction": predict_stump(stump, QUERY)},
        "votes": votes, "vote_prediction": majority(votes),
        "scale_example": {"raw": predict_knn(scale_rows, (0, 0), 1),
                          "y_divided_by_3": predict_knn(scale_rows, (0, 0), 1, (1, 3))},
    }


In [ ]:
import sys
print(sys.executable)
print(sys.version)


## 処理コード

データ、近傍探索、投票、1段決定木、評価の順。`predict`に入力の正解を渡さず、`evaluate`で予測後に答え合わせする。


## 固定例を予想して実行

入力(2,3)：近傍はT4・T2・T3、距離の二乗は1・5・9。k=1はB、k=3はA。木はx≤4でA／B、入力の予測はA。投票B・A・Aの集約はA。


In [ ]:
baseline_result = run_experiment()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))


## 成績の読み方

k=1は学習7/7・検証2/4、k=3は学習6/7・検証4/4。検証でk=3を選び、選択後にテスト4/4を確認する。固定例の結果で、一般的な最適kや実物の性能を示さない。投票は仕組みの例であり、投票構成を検証で選んだ性能比較ではない。

この後は一度に一条件だけ変更する。固定例の`baseline_result`は変更しない。テストを見て変更した場合、同じテストを独立した最終評価と呼ばない。


In [ ]:
changed_query = (3, 3)
for k in (1, 3):
    print(k, [(str(d), i, label) for d, i, label in nearest(TRAIN, changed_query, k)],
          predict_knn(TRAIN, changed_query, k))


## 尺度だけを変更

原点から(0,3,A)と(2,0,B)を比べる。元の距離の二乗は9と4。yを3で割ると1と4となり、k=1の予測はBからAへ。これは指定した変換の例。標準化の尺度を推定する場合は学習側だけで決める。


In [ ]:
scale_rows = (("S1", 0, 3, "A"), ("S2", 2, 0, "B"))
for scales in ((1, 1), (1, 3)):
    print(scales, predict_knn(scale_rows, (0, 0), 1, scales))


## 学習ラベルだけを変更して木を再学習

T4のBをAへ変更した架空の比較例。原データは上書きしない。分岐前はAが3・Bが4、変更後はAが4・Bが3となる。しきい値が同じでも葉の不純度が変わるかを観察する。ラベルを都合よく書き換えて成績を上げる手順ではない。


In [ ]:
changed_train = tuple((i, x, y, "A" if i == "T4" else label)
                      for i, x, y, label in TRAIN)
print("元の木:", fit_stump(TRAIN))
print("比較用の木:", fit_stump(changed_train))


## 振り返り

入力変更による推論と、学習データ変更による再学習を区別できるか。距離順、投票、分岐と不純度から出力の理由を説明する。追加の結果を保存する場合は`programs/outputs/`へ分ける。


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python neighbors_and_trees.py --explain all`で同じ計算表を表示します。各テーマは`--explain neighbors`のように指定できます。従来のコマンドと固定期待値も維持しています。
